In [1]:
import sys
print(sys.executable)

/Users/sarikkoirala/spark-learning/.venv/bin/python


In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder.master("local[*]").appName("exercise1").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/25 14:34:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df = spark.read.csv("data/account.csv", header=True, inferSchema=True)

In [6]:
df.show()

+----------+----------------+-----------+-------+------+
|account_id|            name|       type| region|orders|
+----------+----------------+-----------+-------+------+
|      A100|Green Acres Farm|     Grower|Midwest|   120|
|      A101|  prairie co-op |   Retailer|midwest|   300|
|      A102|   Sunrise Seeds|Distributor|  South|    75|
|      A102|   Sunrise Seeds|Distributor|  South|    75|
|      A103|   Blue Creek Ag|       NULL|   West|  NULL|
+----------+----------------+-----------+-------+------+



In [7]:
df.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- region: string (nullable = true)
 |-- orders: integer (nullable = true)



In [8]:
clean = (df
    .withColumn("name", F.initcap(F.trim("name")))
    .withColumn("region", F.initcap(F.trim("region")))
    .withColumn("type", F.coalesce(F.col("type"), F.lit("Unknown")))
    .withColumn("orders", F.coalesce(F.col("orders"), F.lit(0)))
    .dropDuplicates(["account_id"])
)
clean.show()

+----------+----------------+-----------+-------+------+
|account_id|            name|       type| region|orders|
+----------+----------------+-----------+-------+------+
|      A100|Green Acres Farm|     Grower|Midwest|   120|
|      A101|   Prairie Co-op|   Retailer|Midwest|   300|
|      A102|   Sunrise Seeds|Distributor|  South|    75|
|      A103|   Blue Creek Ag|    Unknown|   West|     0|
+----------+----------------+-----------+-------+------+



In [9]:
clean.write.mode("overwrite").parquet("data/output/accounts_parquet")

In [10]:
spark.read.parquet("data/output/accounts_parquet").show()

+----------+----------------+-----------+-------+------+
|account_id|            name|       type| region|orders|
+----------+----------------+-----------+-------+------+
|      A100|Green Acres Farm|     Grower|Midwest|   120|
|      A101|   Prairie Co-op|   Retailer|Midwest|   300|
|      A102|   Sunrise Seeds|Distributor|  South|    75|
|      A103|   Blue Creek Ag|    Unknown|   West|     0|
+----------+----------------+-----------+-------+------+

